In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
소성가공 품질보증 데이터 기본 분석
기본 라이브러리만 사용하는 간단한 버전
"""

import csv
import math
from collections import defaultdict, Counter
from datetime import datetime


class SimpleQualityAnalyzer:
    """간단한 품질 데이터 분석기"""

    def __init__(self, csv_path):
        self.csv_path = csv_path
        self.data = []
        self.headers = []
        self.feature_stats = defaultdict(lambda: {'values': [], 'min': float('inf'), 'max': float('-inf')})

    def load_data(self):
        """CSV 파일 로드"""
        print("=" * 70)
        print("데이터 로딩 중...")
        print("=" * 70)

        with open(self.csv_path, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            self.headers = next(reader)

            for row in reader:
                # 첫 번째 컬럼은 날짜, 마지막은 레이블
                try:
                    date_str = row[0].strip()
                    features = [float(x) for x in row[1:-1]]
                    label = int(row[-1].strip())

                    self.data.append({
                        'date': date_str,
                        'features': features,
                        'label': label
                    })
                except (ValueError, IndexError) as e:
                    continue

        print(f"\n총 데이터 수: {len(self.data):,}")
        print(f"특성 수: {len(self.headers) - 2}")  # date와 label 제외

        return self

    def basic_statistics(self):
        """기본 통계 분석"""
        print("\n" + "=" * 70)
        print("기본 통계 분석")
        print("=" * 70)

        # 레이블 분포
        labels = [d['label'] for d in self.data]
        label_counts = Counter(labels)

        normal_count = label_counts.get(0, 0)
        defect_count = label_counts.get(1, 0)
        total = len(labels)

        print(f"\n레이블 분포:")
        print(f"  정상 (0): {normal_count:,} ({normal_count / total * 100:.2f}%)")
        print(f"  불량 (1): {defect_count:,} ({defect_count / total * 100:.2f}%)")

        # 특성별 통계
        print(f"\n특성별 통계:")
        feature_names = self.headers[1:-1]  # date와 passorfail 제외

        for idx, feat_name in enumerate(feature_names):
            values = [d['features'][idx] for d in self.data]

            mean_val = sum(values) / len(values)
            sorted_vals = sorted(values)
            median_val = sorted_vals[len(sorted_vals) // 2]
            min_val = min(values)
            max_val = max(values)

            # 표준편차 계산
            variance = sum((x - mean_val) ** 2 for x in values) / len(values)
            std_val = math.sqrt(variance)

            print(f"\n  [{idx + 1:2d}] {feat_name}")
            print(f"       평균: {mean_val:10.3f} | 중앙값: {median_val:10.3f}")
            print(f"       최소: {min_val:10.3f} | 최대: {max_val:10.3f}")
            print(f"       표준편차: {std_val:10.3f}")

    def compare_normal_vs_defect(self):
        """정상 vs 불량 데이터 비교"""
        print("\n" + "=" * 70)
        print("정상 vs 불량 데이터 비교")
        print("=" * 70)

        feature_names = self.headers[1:-1]

        # 데이터 분리
        normal_data = [d for d in self.data if d['label'] == 0]
        defect_data = [d for d in self.data if d['label'] == 1]

        print(f"\n주요 특성 비교 (상위 5개):")

        for idx in range(min(5, len(feature_names))):
            feat_name = feature_names[idx]

            normal_values = [d['features'][idx] for d in normal_data]
            defect_values = [d['features'][idx] for d in defect_data]

            normal_mean = sum(normal_values) / len(normal_values) if normal_values else 0
            defect_mean = sum(defect_values) / len(defect_values) if defect_values else 0

            diff = abs(defect_mean - normal_mean)
            diff_pct = (diff / normal_mean * 100) if normal_mean != 0 else 0

            print(f"\n  [{idx + 1}] {feat_name}")
            print(f"      정상 평균: {normal_mean:10.3f}")
            print(f"      불량 평균: {defect_mean:10.3f}")
            print(f"      차이: {diff:10.3f} ({diff_pct:6.2f}%)")

    def time_analysis(self):
        """시간대별 분석"""
        print("\n" + "=" * 70)
        print("시간대별 불량률 분석")
        print("=" * 70)

        hourly_defects = defaultdict(lambda: {'total': 0, 'defects': 0})

        for d in self.data:
            try:
                # 날짜 문자열에서 시간 추출 (예: "2020 10 30 00:00:04")
                parts = d['date'].split()
                if len(parts) >= 4:
                    hour = int(parts[3].split(':')[0])
                    hourly_defects[hour]['total'] += 1
                    if d['label'] == 1:
                        hourly_defects[hour]['defects'] += 1
            except:
                continue

        print(f"\n시간별 불량률:")
        for hour in sorted(hourly_defects.keys()):
            total = hourly_defects[hour]['total']
            defects = hourly_defects[hour]['defects']
            rate = (defects / total * 100) if total > 0 else 0

            bar_length = int(rate / 2)  # 최대 50칸
            bar = '█' * bar_length

            print(f"  {hour:2d}시: {rate:5.2f}% {bar} ({defects}/{total})")

    def correlation_analysis(self):
        """상관관계 분석 (간단 버전)"""
        print("\n" + "=" * 70)
        print("특성 간 상관관계 분석 (상위 10개)")
        print("=" * 70)

        feature_names = self.headers[1:-1]
        n_features = len(feature_names)

        # 각 특성의 값들을 리스트로 저장
        feature_values = []
        for idx in range(n_features):
            values = [d['features'][idx] for d in self.data]
            feature_values.append(values)

        # 상관계수 계산
        correlations = []

        for i in range(n_features):
            for j in range(i + 1, n_features):
                corr = self._calculate_correlation(feature_values[i], feature_values[j])
                if abs(corr) > 0.7:  # 강한 상관관계만
                    correlations.append((feature_names[i], feature_names[j], corr))

        # 상관계수 절대값 기준 정렬
        correlations.sort(key=lambda x: abs(x[2]), reverse=True)

        print("\n강한 상관관계 (|r| > 0.7):")
        for feat1, feat2, corr in correlations[:10]:
            direction = "양(+)" if corr > 0 else "음(-)"
            print(f"  {feat1[:20]:20s} <-> {feat2[:20]:20s}: {corr:6.3f} ({direction})")

        if not correlations:
            print("  강한 상관관계를 가진 특성 쌍이 없습니다.")

    def _calculate_correlation(self, x, y):
        """피어슨 상관계수 계산"""
        n = len(x)

        mean_x = sum(x) / n
        mean_y = sum(y) / n

        numerator = sum((x[i] - mean_x) * (y[i] - mean_y) for i in range(n))

        sum_sq_x = sum((xi - mean_x) ** 2 for xi in x)
        sum_sq_y = sum((yi - mean_y) ** 2 for yi in y)

        denominator = math.sqrt(sum_sq_x * sum_sq_y)

        if denominator == 0:
            return 0

        return numerator / denominator

    def anomaly_detection_simple(self, threshold_percentile=95):
        """간단한 이상 탐지 (통계 기반)"""
        print("\n" + "=" * 70)
        print("간단한 이상 탐지 (통계 기반)")
        print("=" * 70)

        # 정상 데이터의 통계를 기준으로 사용
        normal_data = [d for d in self.data if d['label'] == 0]

        # 각 특성의 정상 범위 계산
        feature_ranges = []
        for idx in range(len(self.headers) - 2):
            values = [d['features'][idx] for d in normal_data]
            mean_val = sum(values) / len(values)
            variance = sum((x - mean_val) ** 2 for x in values) / len(values)
            std_val = math.sqrt(variance)

            feature_ranges.append({
                'mean': mean_val,
                'std': std_val,
                'lower': mean_val - 3 * std_val,  # 3-sigma 하한
                'upper': mean_val + 3 * std_val  # 3-sigma 상한
            })

        # 테스트 데이터에 대해 이상 점수 계산
        anomaly_scores = []
        for d in self.data:
            score = 0
            for idx, val in enumerate(d['features']):
                r = feature_ranges[idx]
                # 정규화된 거리 계산
                if r['std'] > 0:
                    normalized_dist = abs(val - r['mean']) / r['std']
                    score += normalized_dist

            anomaly_scores.append({
                'score': score,
                'actual_label': d['label']
            })

        # 임계값 설정 (95 백분위수)
        scores = [s['score'] for s in anomaly_scores]
        sorted_scores = sorted(scores)
        threshold_idx = int(len(sorted_scores) * threshold_percentile / 100)
        threshold = sorted_scores[threshold_idx]

        # 예측
        correct = 0
        tp = 0  # True Positive
        tn = 0  # True Negative
        fp = 0  # False Positive
        fn = 0  # False Negative

        for s in anomaly_scores:
            predicted = 1 if s['score'] > threshold else 0
            actual = s['actual_label']

            if predicted == actual:
                correct += 1
                if predicted == 1:
                    tp += 1
                else:
                    tn += 1
            else:
                if predicted == 1:
                    fp += 1
                else:
                    fn += 1

        accuracy = correct / len(anomaly_scores) * 100
        precision = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        print(f"\n이상 탐지 결과:")
        print(f"  임계값: {threshold:.4f}")
        print(f"  정확도: {accuracy:.2f}%")
        print(f"  정밀도: {precision:.2f}%")
        print(f"  재현율: {recall:.2f}%")
        print(f"  F1 Score: {f1:.2f}%")

        print(f"\n혼동 행렬:")
        print(f"                예측")
        print(f"              정상  불량")
        print(f"  실제 정상   {tn:5d} {fp:5d}")
        print(f"      불량   {fn:5d} {tp:5d}")

    def generate_report(self, output_path):
        """분석 리포트 생성"""
        print("\n" + "=" * 70)
        print("분석 리포트 생성 중...")
        print("=" * 70)

        with open(output_path, 'w', encoding='utf-8') as f:
            f.write("=" * 70 + "\n")
            f.write("소성가공 품질보증 데이터 분석 리포트\n")
            f.write("=" * 70 + "\n\n")

            # 기본 정보
            f.write("1. 데이터 개요\n")
            f.write("-" * 70 + "\n")
            f.write(f"총 데이터 수: {len(self.data):,}\n")
            f.write(f"특성 수: {len(self.headers) - 2}\n\n")

            # 레이블 분포
            labels = [d['label'] for d in self.data]
            label_counts = Counter(labels)
            normal_count = label_counts.get(0, 0)
            defect_count = label_counts.get(1, 0)

            f.write("2. 레이블 분포\n")
            f.write("-" * 70 + "\n")
            f.write(f"정상: {normal_count:,} ({normal_count / len(labels) * 100:.2f}%)\n")
            f.write(f"불량: {defect_count:,} ({defect_count / len(labels) * 100:.2f}%)\n\n")

            # 특성 목록
            f.write("3. 특성 목록\n")
            f.write("-" * 70 + "\n")
            for idx, feat_name in enumerate(self.headers[1:-1], 1):
                f.write(f"{idx:2d}. {feat_name}\n")

            f.write("\n분석 완료!\n")

        print(f"리포트 저장: {output_path}")




In [2]:
  """메인 실행 함수"""

  csv_path = '/content/data/2. 소성가공 품질보증 AI 데이터셋.csv'

  # 분석기 생성 및 실행
  analyzer = SimpleQualityAnalyzer(csv_path)
  analyzer.load_data()
  analyzer.basic_statistics()
  analyzer.compare_normal_vs_defect()
  analyzer.time_analysis()
  analyzer.correlation_analysis()
  analyzer.anomaly_detection_simple()

  # 리포트 생성
  report_path = '/content/report/quality_analysis_report.txt'
  analyzer.generate_report(report_path)

  print("\n" + "=" * 70)
  print("모든 분석 완료!")
  print("=" * 70)
  print(f"\n생성된 파일:")
  print(f"  - {report_path}")

데이터 로딩 중...


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/2. 소성가공 품질보증 AI 데이터셋.csv'

In [ ]:
# -*- coding: utf-8 -*-
"""
소성가공 품질보증 AI 데이터셋 분석 - KERAS 3.x 호환 버전
VAE (Variational Autoencoder) 기반 품질 이상 탐지
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda, Dropout, BatchNormalization
from tensorflow.keras.models import Model
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (matplotlib)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False


class QualityDataAnalyzer:
    """소성가공 품질 데이터 분석기"""

    def __init__(self, csv_path):
        self.csv_path = csv_path
        self.data = None
        self.scaler = StandardScaler()
        self.feature_columns = None
        self.target_column = 'passorfail'

    def load_data(self):
        """데이터 로드 및 전처리"""
        print("=" * 60)
        print("데이터 로딩 중...")
        print("=" * 60)

        try:
            self.data = pd.read_csv(self.csv_path)
            print(f"\n원본 데이터 shape: {self.data.shape}")

            missing_info = self.data.isnull().sum()
            if missing_info.sum() > 0:
                print(f"\n결측치 발견:")
                print(missing_info[missing_info > 0])

            try:
                self.data['datetime'] = pd.to_datetime(
                    self.data['date'],
                    format='%Y %m %d %H:%M:%S',
                    errors='coerce'
                )
            except Exception as e:
                print(f"날짜 변환 경고: {e}")
                self.data['datetime'] = pd.to_datetime(self.data['date'], errors='coerce')

            self.feature_columns = [col for col in self.data.columns
                                    if col not in ['date', 'datetime', self.target_column]]

            before_len = len(self.data)
            self.data = self.data.dropna(subset=[self.target_column])
            after_len = len(self.data)

            if before_len != after_len:
                print(f"\n경고: 레이블 결측치 {before_len - after_len}개 행 제거")

            self.data[self.target_column] = self.data[self.target_column].astype(int)

            for col in self.feature_columns:
                if self.data[col].isnull().sum() > 0:
                    mean_val = self.data[col].mean()
                    self.data[col].fillna(mean_val, inplace=True)
                    print(f"  '{col}' 컬럼의 결측치를 평균값({mean_val:.2f})으로 대체")

            inf_mask = np.isinf(self.data[self.feature_columns].values).any(axis=1)
            if inf_mask.sum() > 0:
                print(f"\n무한대 값 {inf_mask.sum()}개 행 제거")
                self.data = self.data[~inf_mask]

            print(f"\n최종 데이터 shape: {self.data.shape}")
            print(f"총 데이터 수: {len(self.data):,}")
            print(f"특성 수: {len(self.feature_columns)}")

            label_counts = self.data[self.target_column].value_counts()
            print(f"\n레이블 분포:")
            print(f"  정상 (0): {label_counts.get(0, 0):,} ({label_counts.get(0, 0) / len(self.data) * 100:.2f}%)")
            print(f"  불량 (1): {label_counts.get(1, 0):,} ({label_counts.get(1, 0) / len(self.data) * 100:.2f}%)")

            return self

        except Exception as e:
            print(f"\n에러 발생: {e}")
            import traceback
            traceback.print_exc()
            raise

    def exploratory_data_analysis(self):
        """탐색적 데이터 분석"""
        print("\n" + "=" * 60)
        print("탐색적 데이터 분석")
        print("=" * 60)

        print("\n기본 통계량:")
        print(self.data[self.feature_columns].describe())

        missing = self.data.isnull().sum()
        if missing.sum() > 0:
            print("\n결측치 정보:")
            print(missing[missing > 0])
        else:
            print("\n결측치: 없음")

        print("\n상관관계가 높은 특성 쌍 (|r| > 0.8):")
        corr_matrix = self.data[self.feature_columns].corr()
        high_corr = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > 0.8:
                    high_corr.append((
                        corr_matrix.columns[i],
                        corr_matrix.columns[j],
                        corr_matrix.iloc[i, j]
                    ))

        if high_corr:
            for feat1, feat2, corr_val in high_corr[:10]:
                print(f"  {feat1} <-> {feat2}: {corr_val:.3f}")
        else:
            print("  없음")

        return self

    def prepare_data(self, test_size=0.2, random_state=42):
        """학습 및 테스트 데이터 준비"""
        print("\n" + "=" * 60)
        print("데이터 분할 및 정규화")
        print("=" * 60)

        X = self.data[self.feature_columns].values
        y = self.data[self.target_column].values

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )

        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        print(f"\n학습 데이터: {X_train_scaled.shape}")
        print(f"  - 정상: {(y_train == 0).sum():,}")
        print(f"  - 불량: {(y_train == 1).sum():,}")
        print(f"\n테스트 데이터: {X_test_scaled.shape}")
        print(f"  - 정상: {(y_test == 0).sum():,}")
        print(f"  - 불량: {(y_test == 1).sum():,}")

        return X_train_scaled, X_test_scaled, y_train, y_test

    def visualize_data_distribution(self, save_path=None):
        """데이터 분포 시각화"""
        print("\n데이터 분포 시각화 중...")

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        label_counts = self.data[self.target_column].value_counts()
        axes[0, 0].bar(['Normal', 'Defective'],
                       [label_counts.get(0, 0), label_counts.get(1, 0)],
                       color=['green', 'red'], alpha=0.7)
        axes[0, 0].set_title('Label Distribution', fontsize=12, fontweight='bold')
        axes[0, 0].set_ylabel('Count')
        axes[0, 0].grid(axis='y', alpha=0.3)

        self.data['hour'] = self.data['datetime'].dt.hour
        defect_by_hour = self.data.groupby('hour')[self.target_column].mean()
        axes[0, 1].plot(defect_by_hour.index, defect_by_hour.values, marker='o', linewidth=2)
        axes[0, 1].set_title('Defect Rate by Hour', fontsize=12, fontweight='bold')
        axes[0, 1].set_xlabel('Hour of Day')
        axes[0, 1].set_ylabel('Defect Rate')
        axes[0, 1].grid(alpha=0.3)

        key_features = self.feature_columns[:4]
        data_for_box = []
        labels_for_box = []
        for feat in key_features:
            data_for_box.append(self.data[feat].values)
            labels_for_box.append(feat)

        axes[1, 0].boxplot(data_for_box, labels=labels_for_box)
        axes[1, 0].set_title('Key Feature Distributions', fontsize=12, fontweight='bold')
        axes[1, 0].set_ylabel('Value')
        axes[1, 0].tick_params(axis='x', rotation=45)
        axes[1, 0].grid(axis='y', alpha=0.3)

        normal_data = self.data[self.data[self.target_column] == 0][self.feature_columns[0]]
        defect_data = self.data[self.data[self.target_column] == 1][self.feature_columns[0]]

        axes[1, 1].hist([normal_data, defect_data], bins=30, label=['Normal', 'Defective'],
                       color=['green', 'red'], alpha=0.6)
        axes[1, 1].set_title(f'{self.feature_columns[0]} Distribution by Label',
                            fontsize=12, fontweight='bold')
        axes[1, 1].set_xlabel('Value')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].legend()
        axes[1, 1].grid(axis='y', alpha=0.3)

        plt.tight_layout()

        if save_path:
            import os
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"시각화 저장: {save_path}")

        plt.close()


# ===== KERAS 3.x 호환 VAE 모델 =====
class VAEQualityDetector(Model):
    """VAE 기반 품질 이상 탐지 모델 (Keras 3.x Subclassing)"""

    def __init__(self, input_dim, latent_dim=8, intermediate_dim=32, **kwargs):
        super(VAEQualityDetector, self).__init__(**kwargs)
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.intermediate_dim = intermediate_dim
        self.history = None

        # Encoder layers
        self.encoder_dense1 = Dense(intermediate_dim * 2, activation='relu', name='enc_dense1')
        self.encoder_bn1 = BatchNormalization(name='enc_bn1')
        self.encoder_dropout1 = Dropout(0.2, name='enc_dropout1')

        self.encoder_dense2 = Dense(intermediate_dim, activation='relu', name='enc_dense2')
        self.encoder_bn2 = BatchNormalization(name='enc_bn2')
        self.encoder_dropout2 = Dropout(0.2, name='enc_dropout2')

        self.z_mean_layer = Dense(latent_dim, name='z_mean')
        self.z_log_var_layer = Dense(latent_dim, name='z_log_var')

        # Decoder layers
        self.decoder_dense1 = Dense(intermediate_dim, activation='relu', name='dec_dense1')
        self.decoder_bn1 = BatchNormalization(name='dec_bn1')
        self.decoder_dropout1 = Dropout(0.2, name='dec_dropout1')

        self.decoder_dense2 = Dense(intermediate_dim * 2, activation='relu', name='dec_dense2')
        self.decoder_bn2 = BatchNormalization(name='dec_bn2')
        self.decoder_dropout2 = Dropout(0.2, name='dec_dropout2')

        self.decoder_output = Dense(input_dim, activation='linear', name='dec_output')

        # Loss trackers
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    def sampling(self, z_mean, z_log_var):
        """Reparameterization trick"""
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    def encode(self, x, training=None):
        """Encoder"""
        h = self.encoder_dense1(x)
        h = self.encoder_bn1(h, training=training)
        h = self.encoder_dropout1(h, training=training)

        h = self.encoder_dense2(h)
        h = self.encoder_bn2(h, training=training)
        h = self.encoder_dropout2(h, training=training)

        z_mean = self.z_mean_layer(h)
        z_log_var = self.z_log_var_layer(h)

        return z_mean, z_log_var

    def decode(self, z, training=None):
        """Decoder"""
        h = self.decoder_dense1(z)
        h = self.decoder_bn1(h, training=training)
        h = self.decoder_dropout1(h, training=training)

        h = self.decoder_dense2(h)
        h = self.decoder_bn2(h, training=training)
        h = self.decoder_dropout2(h, training=training)

        return self.decoder_output(h)

    def call(self, inputs, training=None):
        """Forward pass"""
        z_mean, z_log_var = self.encode(inputs, training=training)
        z = self.sampling(z_mean, z_log_var)
        reconstruction = self.decode(z, training=training)
        return reconstruction

    def train_step(self, data):
        """커스텀 학습 스텝"""
        with tf.GradientTape() as tape:
            z_mean, z_log_var = self.encode(data, training=True)
            z = self.sampling(z_mean, z_log_var)
            reconstruction = self.decode(z, training=True)

            # Reconstruction loss
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(tf.square(data - reconstruction), axis=1)
            )

            # KL divergence loss
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )

            # Total loss
            total_loss = reconstruction_loss + kl_loss

        # Gradients 계산 및 적용
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        # Metrics 업데이트
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def test_step(self, data):
        """검증 스텝"""
        z_mean, z_log_var = self.encode(data, training=False)
        z = self.sampling(z_mean, z_log_var)
        reconstruction = self.decode(z, training=False)

        reconstruction_loss = tf.reduce_mean(
            tf.reduce_sum(tf.square(data - reconstruction), axis=1)
        )

        kl_loss = -0.5 * tf.reduce_mean(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
        )

        total_loss = reconstruction_loss + kl_loss

        return {
            "loss": total_loss,
            "reconstruction_loss": reconstruction_loss,
            "kl_loss": kl_loss,
        }

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def compute_reconstruction_error(self, X):
        """재구성 오차 계산"""
        X_reconstructed = self.predict(X, verbose=0)
        reconstruction_error = np.mean(np.square(X - X_reconstructed), axis=1)
        return reconstruction_error

    def predict_anomaly(self, X, threshold=None):
        """이상 탐지 예측"""
        errors = self.compute_reconstruction_error(X)

        if threshold is None:
            threshold = np.percentile(errors, 95)

        predictions = (errors > threshold).astype(int)
        return predictions, errors, threshold

    def evaluate_model(self, X_test, y_test):
        """모델 평가"""
        print("\n" + "=" * 60)
        print("모델 평가")
        print("=" * 60)

        y_pred, errors, threshold = self.predict_anomaly(X_test)

        print("\n[Classification Report]")
        print(classification_report(y_test, y_pred, target_names=['Normal', 'Defective']))

        cm = confusion_matrix(y_test, y_pred)
        print("\n[Confusion Matrix]")
        print(f"                Predicted")
        print(f"              Normal  Defective")
        print(f"Actual Normal    {cm[0,0]:5d}     {cm[0,1]:5d}")
        print(f"     Defective   {cm[1,0]:5d}     {cm[1,1]:5d}")

        if len(np.unique(y_test)) > 1:
            auc_score = roc_auc_score(y_test, errors)
            print(f"\n[ROC-AUC Score]: {auc_score:.4f}")

        print(f"\n[Threshold]: {threshold:.4f}")

        return y_pred, errors, threshold

    def plot_training_history(self, save_path=None):
        """학습 이력 시각화"""
        if self.history is None:
            print("학습 이력이 없습니다.")
            return

        plt.figure(figsize=(10, 6))
        plt.plot(self.history.history['loss'], label='Training Loss', linewidth=2)
        plt.plot(self.history.history['val_loss'], label='Validation Loss', linewidth=2)
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Loss', fontsize=12)
        plt.title('VAE Training History', fontsize=14, fontweight='bold')
        plt.legend(fontsize=11)
        plt.grid(alpha=0.3)

        if save_path:
            import os
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"학습 이력 저장: {save_path}")

        plt.close()

In [ ]:
  """메인 실행 함수"""

  csv_path = '/content/data/2. 소성가공 품질보증 AI 데이터셋.csv'

  # 1. 데이터 분석
  analyzer = QualityDataAnalyzer(csv_path)
  analyzer.load_data()
  analyzer.exploratory_data_analysis()

  # 2. 데이터 시각화
  analyzer.visualize_data_distribution(save_path='/content/report/data_distribution.png')

  # 3. 데이터 준비
  X_train, X_test, y_train, y_test = analyzer.prepare_data(test_size=0.2, random_state=42)

  # 4. VAE 모델 구축 및 학습
  X_train_normal = X_train[y_train == 0]
  X_test_all = X_test

  print(f"\n정상 데이터만 사용하여 학습: {X_train_normal.shape}")

  # 모델 생성
  vae_detector = VAEQualityDetector(
      input_dim=X_train.shape[1],
      latent_dim=8,
      intermediate_dim=32
  )

  # 모델 컴파일
  vae_detector.compile(optimizer='adam')

  print("\n" + "=" * 60)
  print("VAE 모델 구조")
  print("=" * 60)
  print(f"입력 차원: {X_train.shape[1]}")
  print(f"잠재 차원: 8")
  print(f"중간 레이어: 32")

  # Early Stopping
  early_stopping = tf.keras.callbacks.EarlyStopping(
      monitor='val_loss',
      patience=10,
      restore_best_weights=True,
      verbose=1
  )

  # 학습
  print("\n모델 학습 시작...")
  vae_detector.history = vae_detector.fit(
      X_train_normal,
      epochs=100,
      batch_size=128,
      validation_data=(X_test_all,),
      callbacks=[early_stopping],
      verbose=1
  )

  # 5. 학습 이력 시각화
  vae_detector.plot_training_history(save_path='./report/training_history.png')

  # 6. 모델 평가
  y_pred, errors, threshold = vae_detector.evaluate_model(X_test, y_test)

  # 7. 모델 저장
  import os
  os.makedirs('./model', exist_ok=True)
  vae_detector.save_weights('./model/vae_quality_model.weights.h5')

  print("\n" + "=" * 60)
  print("분석 완료!")
  print("=" * 60)
  print("\n생성된 파일:")
  print("  - report/data_distribution.png")
  print("  - report/training_history.png")
  print("  - model/vae_quality_model.weights.h5")

데이터 로딩 중...

원본 데이터 shape: (17280, 20)

결측치 발견:
passorfail    16
dtype: int64

경고: 레이블 결측치 16개 행 제거

최종 데이터 shape: (17264, 21)
총 데이터 수: 17,264
특성 수: 18

레이블 분포:
  정상 (0): 17,154 (99.36%)
  불량 (1): 110 (0.64%)

탐색적 데이터 분석

기본 통계량:
       EX5.MELT_TEMP  EX4.MELT_TEMP  EX3.MELT_TEMP  EX2.MELT_TEMP  \
count   17264.000000   17264.000000   17264.000000   17264.000000   
mean      296.390697     221.892146     251.020563     266.929275   
std         0.588156       0.369520       0.637041       0.578168   
min       294.000000     219.000000     249.000000     265.000000   
25%       296.000000     222.000000     251.000000     267.000000   
50%       296.000000     222.000000     251.000000     267.000000   
75%       297.000000     222.000000     251.000000     267.000000   
max       300.000000     223.000000     253.000000     269.000000   

          EX1.Z1_PV     EX1.Z2_PV     EX1.Z3_PV     EX1.Z4_PV     EX1.A1_PV  \
count  17264.000000  17264.000000  17264.000000  17264.000000  17264.